In [2]:
import cv2
import numpy as np

# Open the image files
img1_color = cv2.imread("blur1.webp")   # Image to be aligned
img2_color = cv2.imread("blur1.webp")   # Reference image

# Check if images loaded
if img1_color is None or img2_color is None:
    print("Error: Image not found. Check file path.")
else:

    # Convert to grayscale
    img1 = cv2.cvtColor(img1_color, cv2.COLOR_BGR2GRAY)
    img2 = cv2.cvtColor(img2_color, cv2.COLOR_BGR2GRAY)

    height, width = img2.shape

    # Create ORB detector
    orb_detector = cv2.ORB_create(5000)

    # Detect keypoints and descriptors
    kp1, d1 = orb_detector.detectAndCompute(img1, None)
    kp2, d2 = orb_detector.detectAndCompute(img2, None)

    # Safety check
    if d1 is None or d2 is None:
        print("Error: Could not find descriptors in one of the images.")
    else:

        # Brute Force matcher
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

        # Match descriptors
        matches = matcher.match(d1, d2)

        # Convert to list (important fix)
        matches = list(matches)

        # Sort matches by distance
        matches.sort(key=lambda x: x.distance)

        # Take top 90% matches
        matches = matches[:int(len(matches) * 0.9)]
        no_of_matches = len(matches)

        # Create arrays for matched points
        p1 = np.zeros((no_of_matches, 2))
        p2 = np.zeros((no_of_matches, 2))

        for i in range(no_of_matches):
            p1[i, :] = kp1[matches[i].queryIdx].pt
            p2[i, :] = kp2[matches[i].trainIdx].pt

        # Find homography
        homography, mask = cv2.findHomography(p1, p2, cv2.RANSAC)

        # Warp image
        transformed_img = cv2.warpPerspective(
            img1_color, homography, (width, height)
        )

        # Save output
        cv2.imwrite("output.jpg", transformed_img)

        print("Image alignment completed. Saved as output.jpg")

Error: Could not find descriptors in one of the images.
